# 03 - 高级查询（Advanced Queries）
> Harvard CS50’s Intro to Databases with SQL  |  课程时间：1:49:13 – 2:37:37

这章是你的 SQL 分水岭：**子查询**让你在查询里套查询，**JOIN** 让你把多张表的数据拼成一张大结果集，**集合操作**让你像做数学题一样处理数据。

## 学习路线

| # | 内容 |
|---|------|
| 3.1 | 子查询 — 查询里的查询 |
| 3.2 | 嵌套查询进阶 — 相关子查询 |
| 3.3 | IN + 子查询 |
| 3.4 | JOIN — 表与表之间的粘合剂 |
| 3.5 | JOIN 类型 — INNER / LEFT / RIGHT / FULL |
| 3.6 | 集合操作 — UNION / INTERSECT / EXCEPT |

> ⚠️ **先运行 3.0 的 cell 创建数据库！** 本章所有查询都依赖它。

---
## 3.0 准备环境 — 多表数据库

本章需要多张表来演示子查询和 JOIN。我们创建**4 张表**：`authors`（作者）、`publishers`（出版社）、`books`（书）、`ratings`（评分记录）。

In [ ]:
import sqlite3

def sql(query):
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    conn.commit()
    if result is None:
        print('✅ Done')
        return
    if not result:
        print('(empty)')
        return
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')


conn = sqlite3.connect('books_ch3.db')
conn.execute("PRAGMA foreign_keys = ON")

conn.executescript('''
DROP TABLE IF EXISTS ratings;
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS authors;
DROP TABLE IF EXISTS publishers;

CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    birth_year INTEGER,
    country TEXT
);

CREATE TABLE publishers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL UNIQUE,
    country TEXT
);

CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    year INTEGER,
    pages INTEGER,
    genre TEXT,
    author_id INTEGER,
    publisher_id INTEGER,
    FOREIGN KEY (author_id) REFERENCES authors(id),
    FOREIGN KEY (publisher_id) REFERENCES publishers(id)
);

CREATE TABLE ratings (
    id INTEGER PRIMARY KEY,
    book_id INTEGER NOT NULL,
    score REAL NOT NULL,
    review TEXT,
    FOREIGN KEY (book_id) REFERENCES books(id)
);

-- === 数据 ===
INSERT INTO authors VALUES
    (1,  'Stephen King',     1947, 'USA'),
    (2,  'Andy Weir',        1972, 'USA'),
    (3,  'Frank Herbert',    1920, 'USA'),
    (4,  'Gillian Flynn',    1971, 'USA'),
    (5,  'Tara Westover',    1986, 'USA'),
    (6,  'James Clear',      1986, 'USA'),
    (7,  'J.K. Rowling',     1965, 'UK'),
    (8,  'George Orwell',    1903, 'UK'),
    (9,  'Haruki Murakami',  1949, 'Japan'),
    (10, 'Chimamanda Adichie',1977, 'Nigeria'),
    (11, 'Margaret Atwood',  1939, 'Canada'),
    (12, 'Liu Cixin',        1963, 'China');

INSERT INTO publishers VALUES
    (1, 'Doubleday',    'USA'),
    (2, 'Viking',       'USA'),
    (3, 'Crown',        'USA'),
    (4, 'Ballantine',   'USA'),
    (5, 'Chilton',      'USA'),
    (6, 'Random House', 'USA'),
    (7, 'Avery',        'USA'),
    (8, 'Bloomsbury',   'UK'),
    (9, 'Secker',       'UK'),
    (10,'Shinchosha',   'Japan'),
    (11,'McClelland',   'Canada'),
    (12,'Chongqing',    'China');

INSERT INTO books VALUES
    (1,  'The Shining',         1977, 447,  'Horror',          1,  1),
    (2,  'It',                  1986, 1138, 'Horror',          1,  2),
    (3,  'The Stand',           1978, 823,  'Horror',          1,  1),
    (4,  'The Martian',         2011, 369,  'Science Fiction', 2,  3),
    (5,  'Project Hail Mary',   2021, 496,  'Science Fiction', 2,  4),
    (6,  'Dune',                1965, 688,  'Science Fiction', 3,  5),
    (7,  'Gone Girl',           2012, 432,  'Mystery',         4,  3),
    (8,  'Sharp Objects',       2006, 254,  'Mystery',         4,  3),
    (9,  'Educated',            2018, 334,  'Nonfiction',      5,  6),
    (10, 'Atomic Habits',       2018, 320,  'Nonfiction',      6,  7),
    (11, 'Harry Potter 1',      1997, 223,  'Fantasy',         7,  8),
    (12, 'Harry Potter 2',      1998, 251,  'Fantasy',         7,  8),
    (13, '1984',                1949, 328,  'Fiction',         8,  9),
    (14, 'Animal Farm',         1945, 112,  'Fiction',         8,  9),
    (15, 'Norwegian Wood',      1987, 296,  'Fiction',         9,  10),
    (16, '1Q84',                2009, 928,  'Fiction',         9,  10),
    (17, 'Americanah',          2013, 477,  'Fiction',         10, 6),
    (18, 'The Testaments',      2019, 432,  'Fiction',         11, 11),
    (19, 'The Three-Body Problem',2008,400,'Science Fiction',  12, 12),
    (20, 'The Dark Forest',     2008, 512,  'Science Fiction', 12, 12);

-- 评分数据（每本书有 1-3 条评分）
INSERT INTO ratings (book_id, score, review) VALUES
    (1, 4.5, 'Terrifying!'),
    (1, 4.0, NULL),
    (2, 4.2, 'Too long but worth it'),
    (2, 4.5, 'Masterpiece'),
    (4, 4.6, 'Funny and smart'),
    (4, 4.3, 'Great science'),
    (4, 4.2, NULL),
    (5, 4.8, 'Even better than The Martian'),
    (5, 4.6, NULL),
    (6, 4.3, 'Classic'),
    (6, 4.0, 'Dense but rewarding'),
    (7, 4.1, 'Twisty'),
    (7, 4.0, NULL),
    (9, 4.7, 'Inspiring'),
    (9, 4.8, 'Must read'),
    (10,4.5, 'Life-changing'),
    (11,4.4, 'Magical'),
    (11,4.2, NULL),
    (13,4.5, 'Still relevant'),
    (13,4.3, 'A warning'),
    (14,4.2, 'Short but powerful'),
    (15,3.9, 'Melancholic'),
    (17,4.3, 'Eye-opening'),
    (19,4.4, 'Mind-blowing concepts'),
    (20,4.5, 'Dark and philosophical');
''')

print('✅ 数据库创建完毕！')
print(f'   authors:    {conn.execute("SELECT COUNT(*) FROM authors").fetchone()[0]} 人')
print(f'   publishers: {conn.execute("SELECT COUNT(*) FROM publishers").fetchone()[0]} 家')
print(f'   books:      {conn.execute("SELECT COUNT(*) FROM books").fetchone()[0]} 本')
print(f'   ratings:    {conn.execute("SELECT COUNT(*) FROM ratings").fetchone()[0]} 条')
print('\n📋 各表预览：')
print('authors:'); sql('SELECT * FROM authors;')
print('books:');   sql('SELECT id, title, genre, author_id, publisher_id FROM books;')
print('ratings:'); sql('SELECT * FROM ratings;')

---
## 3.1 子查询（Subquery）

**子查询 = 嵌套在 SQL 里的另一个 SELECT。** 括号里的查询先执行，结果交给外层。

In [ ]:
# 子查询最简单的例子：找出「评分高于平均分」的书
# 第 1 步：看看平均分是多少
print('平均评分：')
sql('SELECT ROUND(AVG(score), 2) AS avg_score FROM ratings;')

print('\n评分高于平均分的：')
sql('''SELECT book_id, score
FROM ratings
WHERE score > (SELECT AVG(score) FROM ratings)
ORDER BY score DESC;''')

In [ ]:
# 子查询的三种位置

# ① WHERE 中（最常用）：找 Stephen King 的书
sql('''SELECT title, year
FROM books
WHERE author_id = (SELECT id FROM authors WHERE name = "Stephen King");''')

In [ ]:
# ② SELECT 中：每行都显示总体平均分做参照
sql('''SELECT
    book_id,
    score,
    (SELECT ROUND(AVG(score), 2) FROM ratings) AS overall_avg,
    ROUND(score - (SELECT AVG(score) FROM ratings), 2) AS diff_from_avg
FROM ratings
ORDER BY diff_from_avg DESC
LIMIT 8;''')

In [ ]:
# ③ FROM 中：子查询当作临时表（必须给别名 AS something）
sql('''SELECT genre, avg_pages
FROM (
    SELECT genre, ROUND(AVG(pages)) AS avg_pages
    FROM books
    GROUP BY genre
) AS genre_stats
WHERE avg_pages > 300
ORDER BY avg_pages DESC;''')

### ✏️ 自己动手

用子查询找出「页数超过所有书平均页数」的书名和页数。

In [ ]:
sql("")

---
## 3.2 嵌套查询进阶 — 相关子查询

### 普通子查询 vs 相关子查询

- **普通子查询**：独立执行一次，结果交给外层
- **相关子查询**：依赖于外层的每一行，对每行执行一次

In [ ]:
# 相关子查询：找出「评分高于同类型平均分」的书
# 注意子查询引用了外层 b1.genre
sql('''SELECT title, genre, pages
FROM books AS b1
WHERE pages > (
    SELECT AVG(pages)
    FROM books AS b2
    WHERE b2.genre = b1.genre   -- ← 引用了外层！
)
ORDER BY genre, pages DESC;''')

In [ ]:
# 对于每本书，子查询计算它所属类型的平均页数
# 执行流程：
# b1 = 'The Shining' (Horror, 447p)
#   子查询：AVG(pages) FROM books WHERE genre = 'Horror' → 802.6
#   447 > 802.6? NO → 不输出
# b1 = 'It' (Horror, 1138p)
#   子查询：AVG(pages) FROM books WHERE genre = 'Horror' → 802.6
#   1138 > 802.6? YES → 输出
# ...

### ✏️ 自己动手

用相关子查询找出「评分高于同本书所有评分的平均分」的评分记录。

In [ ]:
sql("")

---
## 3.3 IN + 子查询

子查询返回多行时，用 `IN` 而不是 `=`。

In [ ]:
# 找「1970 年之后出生的作者写的书」
# 子查询返回多个 ID → 必须用 IN
sql('''SELECT title, author_id
FROM books
WHERE author_id IN (
    SELECT id FROM authors WHERE birth_year > 1970
)
ORDER BY title;''')

In [ ]:
# NOT IN：找「没有评分记录的书」
sql('''SELECT id, title
FROM books
WHERE id NOT IN (
    SELECT DISTINCT book_id FROM ratings
)
ORDER BY title;''')

In [ ]:
# 多级嵌套：找「UK 作者写的、有评分记录的书」
sql('''SELECT DISTINCT b.title, a.name, a.country
FROM books b
JOIN authors a ON b.author_id = a.id
WHERE b.id IN (
    SELECT book_id FROM ratings
)
AND b.author_id IN (
    SELECT id FROM authors WHERE country = 'UK'
)
ORDER BY b.title;''')

### ✏️ 自己动手

找出「页数超过 500 的书所属的作者名字」。用 `IN` 子查询。

In [ ]:
sql("")

---
## 3.4 JOIN 入门

### 为什么 JOIN？

`books` 表只存了 `author_id`（一个数字），但人类要看的是作者**名字**。JOIN 把两张表'粘'在一起。

In [ ]:
# 没有 JOIN 的时代（假想）：先查出 author_id，再手动去查
sql("SELECT title, author_id FROM books LIMIT 3;")
print('author_id = 1 是谁？还得再查一次...')
sql("SELECT name FROM authors WHERE id = 1;")
print('\n🤯 每次都要这样手动查！JOIN 一步到位 ↓')

In [ ]:
# INNER JOIN：一步到位，书名 + 作者名
sql('''SELECT b.title, a.name AS author
FROM books b
INNER JOIN authors a ON b.author_id = a.id
LIMIT 5;''')

In [ ]:
# 三表 JOIN：书名 + 作者 + 出版社
sql('''SELECT b.title, a.name AS author, p.name AS publisher
FROM books b
JOIN authors a ON b.author_id = a.id
JOIN publishers p ON b.publisher_id = p.id
LIMIT 8;''')

In [ ]:
# 甚至可以 JOIN 四张表：加上评分
sql('''SELECT b.title, a.name AS author, r.score, r.review
FROM books b
JOIN authors a ON b.author_id = a.id
JOIN ratings r ON b.id = r.book_id
WHERE r.score >= 4.5
ORDER BY r.score DESC;''')

### JOIN 要点

```sql
FROM 表1
JOIN 表2 ON 表1.列 = 表2.列   -- ON 是连接条件（通常是 PK = FK）
JOIN 表3 ON ...               -- 可以链式 JOIN
```

- `b`, `a`, `p` 是表别名（Table Alias），让 SQL 更短、更清晰
- `ON` 后面的条件是逐行匹配的规则

---
## 3.5 JOIN 类型

来看四种 JOIN 的区别。我们先看两张小表，直观理解：

In [ ]:
# 注意看数据：Orwell (id=8) 有书，但 Atwood (id=11) 的书 The Testaments
# 没有评分记录。另外 books 里有些书也没有评分。
print('authors 中有，books 中也有的：')
sql('''SELECT a.id, a.name, b.title
FROM authors a
INNER JOIN books b ON a.id = b.author_id
WHERE a.id IN (8, 11, 12);''')

In [ ]:
# ===== INNER JOIN：只返回两表匹配的行 =====
print('INNER JOIN —— 两表都有才算：')
sql('''SELECT a.name, b.title
FROM authors a
INNER JOIN books b ON a.id = b.author_id
ORDER BY a.name
LIMIT 10;''')

In [ ]:
# ===== LEFT JOIN：左表全部保留，右表没匹配就填 NULL =====
# 「列出所有作者，有书的显示书名，没书的也显示出来」
sql('''SELECT a.name, b.title
FROM authors a
LEFT JOIN books b ON a.id = b.author_id
ORDER BY b.title NULLS LAST;''')

In [ ]:
# LEFT JOIN 最经典用途：找「左表有但右表没有的」
# 哪些作者没有写书？
sql('''SELECT a.name, a.country
FROM authors a
LEFT JOIN books b ON a.id = b.author_id
WHERE b.id IS NULL;''')

In [ ]:
# ===== RIGHT JOIN =====
# SQLite 不支持 RIGHT JOIN，但用 LEFT JOIN 反过来就行
print('SQLite 不支持 RIGHT JOIN，不过 LEFT 反过来就是 RIGHT')

In [ ]:
# FULL JOIN 模拟（SQLite 也不直接支持）
# 用 LEFT JOIN + UNION + RIGHT JOIN 技巧
# 找出「所有有书的作者 + 所有有评分的 book_id」
print('FULL JOIN = 两表全部保留（模拟）：')
sql('''SELECT a.name, b.title
FROM authors a
LEFT JOIN books b ON a.id = b.author_id
UNION
SELECT a.name, b.title
FROM books b
LEFT JOIN authors a ON a.id = b.author_id;''')

### 四种 JOIN 直观对照

```
authors (A)         books (B)
┌────┬────────┐    ┌────┬───────────┬────┐
│ id │ name   │    │ id │ title     │a_id│
├────┼────────┤    ├────┼───────────┼────┤
│ 7  │Rowling │    │ 11 │ HP1       │ 7  │
│ 8  │Orwell  │    │ 13 │ 1984      │ 8  │
│ 11 │Atwood  │    │ 18 │Testaments │ 11 │
│ 12 │Liu     │    │ 19 │3-Body     │ 12 │
│ 2  │Weir    │    │ 4  │Martian    │ 2  │
│ -- │ --     │    │ 20 │Dark F.    │ 12 │
└────┴────────┘    │ 5  │Hail Mary  │ 2  │
      6 行          │ ...│ ...       │ ...│
                    └────┴───────────┴────┘
                         20 行

INNER JOIN (A ∩ B)       LEFT JOIN (A ⊂ B)
= 两表都有的行            = A 全部 + B 匹配部分
  11 行（11 个作者有书）   包含没有书的作者（NULL）
```

### ✏️ 自己动手

用 LEFT JOIN 找出「哪些出版社没有出版过书」。

In [ ]:
sql("")

---
## 3.6 集合操作

把两个查询结果当**数学集合**来运算。

In [ ]:
# UNION：合并两个结果集（自动去重）
# 「2000 年之前出版的书」+「所有 Sci-Fi 书」
sql('''SELECT title, year, 'Old' AS source
FROM books WHERE year < 2000
UNION
SELECT title, year, 'Sci-Fi' AS source
FROM books WHERE genre = 'Science Fiction'
ORDER BY year;''')

In [ ]:
# UNION ALL：合并但不去重（更快，保留重复）
sql('''SELECT title FROM books WHERE year < 2000
UNION ALL
SELECT title FROM books WHERE genre = 'Science Fiction'
ORDER BY title;''')

In [ ]:
# INTERSECT：交集 — 两个查询都有的行
# 「2000 年之前出版」AND「Science Fiction」= 老科幻小说
sql('''SELECT title, year, genre
FROM books WHERE year < 2000
INTERSECT
SELECT title, year, genre
FROM books WHERE genre = 'Science Fiction'
ORDER BY year;''')

In [ ]:
# EXCEPT：差集 — 在第一个查询中但不在第二个中的
# 「2000 年之前的书」-「Science Fiction」= 非科幻的老书
sql('''SELECT title, year, genre
FROM books WHERE year < 2000
EXCEPT
SELECT title, year, genre
FROM books WHERE genre = 'Science Fiction'
ORDER BY year;''')

### 集合操作规则

1. 两个 SELECT 的列数必须相同
2. 对应列类型兼容
3. `UNION` 去重，`UNION ALL` 不去重（后者更快）
4. `INTERSECT` / `EXCEPT` 自动去重

### ✏️ 自己动手

用 EXCEPT 找出「有评分但页数不超过 300」的书。（提示：有评分的书 EXCEPT 页数超过 300 的书）

In [ ]:
sql("")

---
## 🎯 综合挑战

综合运用子查询 + JOIN + 集合操作。

In [ ]:
# Q1：查出每本书的书名 + 作者名 + 平均评分
#     提示：JOIN books + authors，子查询算 AVG(score)
sql("")

In [ ]:
# Q2：找出「同一作者写了 2 本以上书」的作者名字
#     提示：子查询 + GROUP BY + HAVING COUNT(*) >= 2
sql("")

In [ ]:
# Q3：列出所有作者，及他们写的书的数量（没写书的显示 0）
#     提示：LEFT JOIN + COUNT + GROUP BY
sql("")

In [ ]:
# Q4：同时满足「USA 作者写的」且「评分 ≥ 4.5」的书
#     提示：INTERSECT 或 JOIN
sql("")

In [ ]:
# Q5：哪本书有最多的评分记录？显示书名和评分数量
#     提示：JOIN + COUNT + GROUP BY + ORDER BY + LIMIT
sql("")

---
## ✅ 检查清单

- [ ] 能写 WHERE 中的子查询（返回单值用 =，多值用 IN）
- [ ] 理解 SELECT 和 FROM 中的子查询
- [ ] 知道相关子查询 vs 普通子查询的区别
- [ ] 能用 IN + 子查询做多级过滤
- [ ] 能用 INNER JOIN 跨多表查询数据
- [ ] 能用 LEFT JOIN 保留左表全部行
- [ ] 知道四种 JOIN 的区别和适用场景
- [ ] 能用 UNION / INTERSECT / EXCEPT 做集合运算
- [ ] 知道 UNION vs UNION ALL 的去重区别
- [ ] 能用表别名简化多表查询

---

> 📖 下一章：[04 - Schema 与数据操作](../04-schema-and-data-manipulation/) — GROUP BY / HAVING、范式化、建表与 CRUD